
# Machine Learning — Massive Bank Dataset (1 Juta Baris)
## Pemodelan dengan PySpark MLlib di Google Colab

**Dataset**: [Massive Bank Dataset – 1 Million Rows (Kaggle)](https://www.kaggle.com/datasets/ksabishek/massive-bank-dataset-1-million-rows)  

### Model yang Dibangun:
| # | Model | Tipe | Target |
|---|-------|------|--------|
| 1 | **Linear Regression** | Regresi | Prediksi nilai transaksi (`Value`) |
| 2 | **Random Forest Regressor** | Regresi | Prediksi nilai transaksi (`Value`) |
| 3 | **Logistic Regression** | Klasifikasi | Prediksi domain transaksi (`Domain`) |
| 4 | **Random Forest Classifier** | Klasifikasi | Prediksi domain transaksi (`Domain`) |
| 5 | **K-Means Clustering** | Unsupervised | Segmentasi pola transaksi |

---



## Inisialisasi Environment

In [ ]:
!pip install pyspark --quiet
print("PySpark berhasil diinstall")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# MLlib — Feature Engineering
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    StandardScaler, MinMaxScaler
)
from pyspark.ml import Pipeline

# MLlib — Models
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans

# MLlib — Evaluation
from pyspark.ml.evaluation import (
    RegressionEvaluator, MulticlassClassificationEvaluator, ClusteringEvaluator
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
import time, os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

print('Semua library berhasil diimpor')

In [ ]:
spark = SparkSession.builder \
    .appName('BankML') \
    .config('spark.sql.repl.eagerEval.enabled', True) \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

print(f'SparkSession aktif — versi {spark.version}')

## 1. load & prsiapkan data

In [ ]:
csv_path = '/content/bankdataset.csv'

df_raw = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .option('sep', ';') \
    .csv(csv_path)

print(f'Data dimuat: {df_raw.count():,} baris')
df_raw.printSchema()

## 2. Preprocessing & Feature Engineering


In [ ]:
from pyspark.sql.functions import to_date

df = df_raw \
    .dropna() \
    .dropDuplicates() \
    .withColumn('Date',    F.to_date(F.col('Date'), 'dd/MM/yyyy')) \
    .withColumn('Year',    F.year('Date')) \
    .withColumn('Month',   F.month('Date')) \
    .withColumn('Quarter', F.quarter('Date')) \
    .withColumn('DayOfWeek', F.dayofweek('Date')) \
    .withColumn('DayOfYear',  F.dayofyear('Date')) \
    .withColumn('Domain',  F.upper(F.trim(F.col('Domain')))) \
    .withColumn('Transaction_Count', F.col('Transaction_count').cast('double')) \
    .withColumn('Value',   F.col('Value').cast('double')) \
    .withColumn('Avg_Value_per_Txn', F.round(F.col('Value') / F.col('Transaction_Count'), 2)) \
    .withColumn('Log_Value', F.log1p(F.col('Value')))  # log transform untuk regresi

df = df.filter(F.col('Date').isNotNull())

# Tambah fitur agregat per kota (city-level stats)
city_stats = df.groupBy('Location').agg(
    F.avg('Value').alias('City_Avg_Value'),
    F.count('*').alias('City_Record_Count')
)
df = df.join(city_stats, on='Location', how='left')

df.cache()
print(f'Preprocessing selesai: {df.count():,} baris bersih')
df.select('Date','Domain','Location','Value','Year','Month','Quarter',
           'DayOfWeek','Avg_Value_per_Txn','Log_Value','City_Avg_Value').show(5, truncate=False)


##Exploratory Data Analysis (Sebelum Modeling)

In [ ]:
# Distribusi nilai transaksi (Value)
df_val = df.select('Value', 'Domain').sample(fraction=0.05, seed=42).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(df_val['Value'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Nilai Transaksi (Value)', fontweight='bold')
axes[0].set_xlabel('Value (Rupee)')
axes[0].set_ylabel('Frekuensi')

order = df_val.groupby('Domain')['Value'].median().sort_values(ascending=False).index
sns.boxplot(data=df_val, x='Domain', y='Value', order=order, ax=axes[1], palette='Blues')
axes[1].set_title('Distribusi Value per Domain', fontweight='bold')
axes[1].set_xlabel('Domain')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('/content/ml_eda_distribution.png', dpi=150)
plt.show()
print('EDA — distribusi nilai transaksi')

In [ ]:
# Korelasi fitur numerik
num_cols = ['Value', 'Transaction_Count', 'Month', 'Quarter', 'DayOfWeek', 'City_Avg_Value']
df_corr = df.select(num_cols).sample(fraction=0.1, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(8, 6))
corr = df_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, ax=ax, linewidths=0.5)
ax.set_title('Matriks Korelasi Fitur Numerik', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('/content/ml_correlation_matrix.png', dpi=150)
plt.show()

##Analisis dan Agregasi Spark SQL

In [ ]:
# Mendaftarkan DataFrame sebagai tabel virtual
df.createOrReplaceTempView("bank_transactions")
print("TempView 'bank_transactions' berhasil dibuat")

In [ ]:
# Query 1: Total dan rata-rata transaksi per domain bisnis
result_q1 = spark.sql("""
    SELECT
        Domain,
        COUNT(*) AS total_record,
        ROUND(SUM(Value), 0) AS total_value,
        ROUND(AVG(Value), 2) AS avg_value,
        ROUND(AVG(Transaction_Count), 1) AS avg_txn_count
    FROM bank_transactions
    GROUP BY Domain
    ORDER BY total_value DESC
""")
print("=== Query 1: Performa Transaksi per Domain ===")
result_q1.show(truncate=False)

In [ ]:
# Query 2: Tren nilai transaksi per bulan dan kuartal
result_q2 = spark.sql("""
    SELECT
        Year,
        Quarter,
        Month,
        COUNT(*) AS jumlah_transaksi,
        ROUND(SUM(Value), 0) AS total_value,
        ROUND(AVG(Value), 2) AS avg_value
    FROM bank_transactions
    GROUP BY Year, Quarter, Month
    ORDER BY Year, Month
""")
print("=== Query 2: Tren Transaksi Bulanan ===")
result_q2.show(20, truncate=False)

In [ ]:
# Query 3: Kota dengan total nilai transaksi tertinggi
result_q3 = spark.sql("""
    SELECT
        Location,
        COUNT(*) AS jumlah_record,
        ROUND(SUM(Value), 0) AS total_value,
        ROUND(AVG(Value), 2) AS avg_value,
        ROUND(MAX(Value), 0) AS max_value
    FROM bank_transactions
    GROUP BY Location
    ORDER BY total_value DESC
    LIMIT 10
""")
print("=== Query 3: Top 10 Kota dengan Transaksi Terbesar ===")
result_q3.show(truncate=False)

In [ ]:
# Query 4: Segmentasi transaksi berdasarkan kategori nilai
result_q4 = spark.sql("""
    SELECT
        Value_Category,
        Domain,
        COUNT(*) AS jumlah_transaksi,
        ROUND(AVG(Value), 2) AS avg_value
    FROM (
        SELECT *,
            CASE
                WHEN Value < 10000  THEN 'Rendah (<10K)'
                WHEN Value < 50000  THEN 'Menengah (10K-50K)'
                WHEN Value < 100000 THEN 'Tinggi (50K-100K)'
                ELSE 'Premium (>100K)'
            END AS Value_Category
        FROM bank_transactions
    ) t
    GROUP BY Value_Category, Domain
    ORDER BY Value_Category, jumlah_transaksi DESC
""")
print("=== Query 4: Segmentasi Nilai Transaksi per Domain ===")
result_q4.show(30, truncate=False)

In [ ]:
# Query 5: Pola transaksi berdasarkan hari dalam seminggu
result_q5 = spark.sql("""
    SELECT
        DayOfWeek,
        CASE DayOfWeek
            WHEN 1 THEN 'Minggu'
            WHEN 2 THEN 'Senin'
            WHEN 3 THEN 'Selasa'
            WHEN 4 THEN 'Rabu'
            WHEN 5 THEN 'Kamis'
            WHEN 6 THEN 'Jumat'
            WHEN 7 THEN 'Sabtu'
        END AS Nama_Hari,
        COUNT(*) AS jumlah_transaksi,
        ROUND(AVG(Value), 2) AS avg_value,
        ROUND(SUM(Value), 0) AS total_value
    FROM bank_transactions
    GROUP BY DayOfWeek
    ORDER BY DayOfWeek
""")
print("=== Query 5: Pola Transaksi per Hari ===")
result_q5.show(truncate=False)

## 3. MODEL 1 — Linear Regression
**Target**: Prediksi `Value` (nilai transaksi)  
**Fitur**: Month, Quarter, DayOfWeek, Transaction_Count, Domain (encoded), City_Avg_Value

In [ ]:
# FEATURE ENGINEERING PIPELINE — REGRESI
domain_indexer  = StringIndexer(inputCol='Domain',   outputCol='Domain_Idx',   handleInvalid='keep')
location_indexer = StringIndexer(inputCol='Location', outputCol='Location_Idx', handleInvalid='keep')
domain_ohe      = OneHotEncoder(inputCols=['Domain_Idx'],   outputCols=['Domain_Vec'])
location_ohe    = OneHotEncoder(inputCols=['Location_Idx'], outputCols=['Location_Vec'])

feature_cols_reg = [
    'Month', 'Quarter', 'DayOfWeek', 'Year',
    'Transaction_Count', 'City_Avg_Value',
    'Domain_Vec', 'Location_Vec'
]

assembler_reg = VectorAssembler(
    inputCols=feature_cols_reg,
    outputCol='features_raw',
    handleInvalid='skip'
)
scaler_reg = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withMean=False, withStd=True
)

# Siapkan data regresi
df_reg = df.withColumnRenamed('Value', 'label')
train_reg, test_reg = df_reg.randomSplit([0.8, 0.2], seed=42)

print(f'Train: {train_reg.count():,} | Test: {test_reg.count():,}')

In [ ]:
# MODEL 1: LINEAR REGRESSION
lr = LinearRegression(
    featuresCol='features',
    labelCol='label',
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.5  # ElasticNet (mix L1+L2)
)

pipeline_lr = Pipeline(stages=[
    domain_indexer, location_indexer,
    domain_ohe, location_ohe,
    assembler_reg, scaler_reg, lr
])

print('Training Linear Regression...')
t0 = time.time()
model_lr = pipeline_lr.fit(train_reg)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

# Evaluasi
pred_lr = model_lr.transform(test_reg)
evaluator_reg = RegressionEvaluator(labelCol='label', predictionCol='prediction')

rmse_lr = evaluator_reg.evaluate(pred_lr, {evaluator_reg.metricName: 'rmse'})
mae_lr  = evaluator_reg.evaluate(pred_lr, {evaluator_reg.metricName: 'mae'})
r2_lr   = evaluator_reg.evaluate(pred_lr, {evaluator_reg.metricName: 'r2'})

print(f'\nLinear Regression — Hasil Evaluasi:')
print(f'   RMSE : {rmse_lr:,.2f}')
print(f'   MAE  : {mae_lr:,.2f}')
print(f'   R²   : {r2_lr:.4f}')

In [ ]:
# Visualisasi: Actual vs Predicted
sample_lr = pred_lr.select('label', 'prediction').sample(fraction=0.02, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sample_lr['label'], sample_lr['prediction'],
           alpha=0.4, s=10, color='steelblue')
lims = [min(sample_lr['label'].min(), sample_lr['prediction'].min()),
        max(sample_lr['label'].max(), sample_lr['prediction'].max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual Value')
ax.set_ylabel('Predicted Value')
ax.set_title(f'Linear Regression — Actual vs Predicted\nR² = {r2_lr:.4f}  |  RMSE = {rmse_lr:,.0f}',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/content/ml_lr_actual_vs_predicted.png', dpi=150)
plt.show()

## 4. MODEL 2 — Random Forest Regressor
**Target**: Prediksi `Value`  
Model ensemble — biasanya lebih akurat dari Linear Regression untuk data non-linear.

In [ ]:
# MODEL 2: RANDOM FOREST REGRESSOR
rf_reg = RandomForestRegressor(
    featuresCol='features',
    labelCol='label',
    numTrees=100,
    maxDepth=8,
    seed=42
)

pipeline_rf_reg = Pipeline(stages=[
    domain_indexer, location_indexer,
    domain_ohe, location_ohe,
    assembler_reg, scaler_reg, rf_reg
])

print('Training Random Forest Regressor (numTrees=100)...')
t0 = time.time()
model_rf_reg = pipeline_rf_reg.fit(train_reg)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

pred_rf_reg = model_rf_reg.transform(test_reg)

rmse_rf = evaluator_reg.evaluate(pred_rf_reg, {evaluator_reg.metricName: 'rmse'})
mae_rf  = evaluator_reg.evaluate(pred_rf_reg, {evaluator_reg.metricName: 'mae'})
r2_rf   = evaluator_reg.evaluate(pred_rf_reg, {evaluator_reg.metricName: 'r2'})

print(f'\n Random Forest Regressor — Hasil Evaluasi:')
print(f'   RMSE : {rmse_rf:,.2f}')
print(f'   MAE  : {mae_rf:,.2f}')
print(f'   R²   : {r2_rf:.4f}')

In [ ]:
# Feature Importance — Random Forest Regressor
rf_model_stage = model_rf_reg.stages[-1]  # ambil RF stage
importances = rf_model_stage.featureImportances.toArray()

# Label fitur (sederhana — numerik saja karena OHE expand)
base_features = ['Month', 'Quarter', 'DayOfWeek', 'Year',
                 'Transaction_Count', 'City_Avg_Value']
n_base = len(base_features)
labels = base_features + [f'Domain_OHE_{i}' for i in range(7)] + \
         [f'Location_OHE_{i}' for i in range(min(50, len(importances)-n_base-7))]
labels = labels[:len(importances)]

fi_df = pd.DataFrame({'Feature': labels, 'Importance': importances[:len(labels)]})
fi_df = fi_df.sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=fi_df, x='Importance', y='Feature', palette='Blues_d', ax=ax)
ax.set_title('Feature Importance — Random Forest Regressor (Top 15)', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('/content/ml_rf_feature_importance.png', dpi=150)
plt.show()

## 5. MODEL 3 — Logistic Regression (Klasifikasi Domain)
**Target**: Prediksi `Domain` transaksi (7 kelas)  
**Fitur**: Month, Quarter, Value, Transaction_Count, City_Avg_Value, Location (encoded)

In [ ]:
# PERSIAPAN DATA KLASIFIKASI
label_indexer = StringIndexer(
    inputCol='Domain', outputCol='label', handleInvalid='keep'
)

location_idx2 = StringIndexer(
    inputCol='Location', outputCol='Location_Idx', handleInvalid='keep'
)
location_ohe2 = OneHotEncoder(
    inputCols=['Location_Idx'], outputCols=['Location_Vec']
)

feature_cols_cls = [
    'Month', 'Quarter', 'DayOfWeek', 'Year',
    'Value', 'Transaction_Count', 'City_Avg_Value',
    'Location_Vec'
]

assembler_cls = VectorAssembler(
    inputCols=feature_cols_cls,
    outputCol='features_raw',
    handleInvalid='skip'
)
scaler_cls = StandardScaler(
    inputCol='features_raw', outputCol='features',
    withMean=False, withStd=True
)

train_cls, test_cls = df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_cls.count():,} | Test: {test_cls.count():,}')

In [ ]:
# MODEL 3: LOGISTIC REGRESSION (Multiclass)
log_reg = LogisticRegression(
    featuresCol='features',
    labelCol='label',
    maxIter=100,
    regParam=0.1,
    family='multinomial'
)

pipeline_logreg = Pipeline(stages=[
    label_indexer, location_idx2, location_ohe2,
    assembler_cls, scaler_cls, log_reg
])

print('Training Logistic Regression (Multiclass)...')
t0 = time.time()
model_logreg = pipeline_logreg.fit(train_cls)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

pred_logreg = model_logreg.transform(test_cls)

evaluator_cls = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction'
)

acc_lr  = evaluator_cls.evaluate(pred_logreg, {evaluator_cls.metricName: 'accuracy'})
f1_lr   = evaluator_cls.evaluate(pred_logreg, {evaluator_cls.metricName: 'f1'})
prec_lr = evaluator_cls.evaluate(pred_logreg, {evaluator_cls.metricName: 'weightedPrecision'})
rec_lr  = evaluator_cls.evaluate(pred_logreg, {evaluator_cls.metricName: 'weightedRecall'})

print(f'\nLogistic Regression — Hasil Evaluasi:')
print(f'   Accuracy  : {acc_lr:.4f} ({acc_lr*100:.2f}%)')
print(f'   F1 Score  : {f1_lr:.4f}')
print(f'   Precision : {prec_lr:.4f}')
print(f'   Recall    : {rec_lr:.4f}')

## 6. MODEL 4 — Random Forest Classifier
**Target**: Prediksi `Domain` transaksi

In [ ]:
# MODEL 4: RANDOM FOREST CLASSIFIER
rf_cls = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    numTrees=100,
    maxDepth=8,
    seed=42
)

pipeline_rfc = Pipeline(stages=[
    label_indexer, location_idx2, location_ohe2,
    assembler_cls, scaler_cls, rf_cls
])

print('Training Random Forest Classifier...')
t0 = time.time()
model_rfc = pipeline_rfc.fit(train_cls)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

pred_rfc = model_rfc.transform(test_cls)

acc_rfc  = evaluator_cls.evaluate(pred_rfc, {evaluator_cls.metricName: 'accuracy'})
f1_rfc   = evaluator_cls.evaluate(pred_rfc, {evaluator_cls.metricName: 'f1'})
prec_rfc = evaluator_cls.evaluate(pred_rfc, {evaluator_cls.metricName: 'weightedPrecision'})
rec_rfc  = evaluator_cls.evaluate(pred_rfc, {evaluator_cls.metricName: 'weightedRecall'})

print(f'\nRandom Forest Classifier — Hasil Evaluasi:')
print(f'   Accuracy  : {acc_rfc:.4f} ({acc_rfc*100:.2f}%)')
print(f'   F1 Score  : {f1_rfc:.4f}')
print(f'   Precision : {prec_rfc:.4f}')
print(f'   Recall    : {rec_rfc:.4f}')

In [ ]:
# Confusion Matrix — Random Forest Classifier
labels_list = model_rfc.stages[0].labels

cm_df = pred_rfc.groupBy('label', 'prediction').count().toPandas()
n_cls = len(labels_list)
cm = np.zeros((n_cls, n_cls), dtype=int)
for _, row in cm_df.iterrows():
    i, j = int(row['label']), int(row['prediction'])
    if i < n_cls and j < n_cls:
        cm[i][j] = row['count']

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_list, yticklabels=labels_list, ax=ax)
ax.set_xlabel('Predicted Domain', fontsize=11)
ax.set_ylabel('Actual Domain', fontsize=11)
ax.set_title(f'Confusion Matrix — Random Forest Classifier\nAccuracy: {acc_rfc*100:.2f}%',
             fontweight='bold', fontsize=12)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('/content/ml_rfc_confusion_matrix.png', dpi=150)
plt.show()

## 7. MODEL 5 — K-Means Clustering
**Tujuan**: Segmentasi transaksi ke dalam klaster berdasarkan pola Value, Transaction_Count, Month  
Model unsupervised — tidak butuh label, menemukan pola sendiri.

In [ ]:
# PERSIAPAN DATA CLUSTERING
feature_cols_km = ['Value', 'Transaction_Count', 'Month', 'Quarter', 'City_Avg_Value']

assembler_km = VectorAssembler(
    inputCols=feature_cols_km,
    outputCol='features_raw',
    handleInvalid='skip'
)
scaler_km = MinMaxScaler(
    inputCol='features_raw', outputCol='features'
)

# Siapkan data
prep_pipeline = Pipeline(stages=[assembler_km, scaler_km])
df_km = prep_pipeline.fit(df).transform(df)
df_km.cache()

# ELBOW METHOD — Tentukan jumlah klaster optimal
print('Mencari jumlah klaster optimal (Elbow Method)...')
k_values = range(2, 9)
wcss = []

for k in k_values:
    km = KMeans(featuresCol='features', k=k, seed=42, maxIter=30)
    km_model = km.fit(df_km)
    wcss.append(km_model.summary.trainingCost)
    print(f'   k={k} → WCSS={km_model.summary.trainingCost:.4f}')

# Plot Elbow
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_values), wcss, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Jumlah Klaster (k)')
ax.set_ylabel('WCSS (Within-Cluster Sum of Squares)')
ax.set_title('Elbow Method — Menentukan k Optimal', fontweight='bold')
ax.set_xticks(list(k_values))
plt.tight_layout()
plt.savefig('/content/ml_kmeans_elbow.png', dpi=150)
plt.show()
print('Elbow plot selesai — pilih k di titik siku grafik')

In [ ]:
# MODEL 5: K-MEANS dengan k optimal (misalnya k=4)
K_OPTIMAL = 4  # Ganti sesuai hasil elbow plot

kmeans = KMeans(featuresCol='features', k=K_OPTIMAL, seed=42, maxIter=50)

print(f'Training K-Means (k={K_OPTIMAL})...')
t0 = time.time()
model_km = kmeans.fit(df_km)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

df_clustered = model_km.transform(df_km)

# Evaluasi Silhouette Score
evaluator_km = ClusteringEvaluator(
    featuresCol='features', predictionCol='prediction', metricName='silhouette'
)
silhouette = evaluator_km.evaluate(df_clustered)
print(f'\n K-Means (k={K_OPTIMAL}) — Silhouette Score: {silhouette:.4f}')
print('   (Skor mendekati 1.0 = klaster terpisah dengan baik)')

In [ ]:
# Profil tiap klaster
print('Profil Klaster:')
cluster_profile = df_clustered.groupBy('prediction').agg(
    F.count('*').alias('Jumlah_Record'),
    F.round(F.avg('Value'), 0).alias('Avg_Value'),
    F.round(F.avg('Transaction_Count'), 0).alias('Avg_Txn_Count'),
    F.round(F.avg('Month'), 1).alias('Avg_Month')
).orderBy('prediction')
cluster_profile.show()

# Domain paling dominan per klaster
print('\n Domain Dominan per Klaster:')
df_clustered.groupBy('prediction', 'Domain').count() \
    .orderBy('prediction', F.desc('count')) \
    .show(K_OPTIMAL * 3)

In [ ]:
# Visualisasi Klaster (Value vs Transaction_Count)
sample_km = df_clustered.select(
    'Value', 'Transaction_Count', 'prediction', 'Domain'
).sample(fraction=0.05, seed=42).toPandas()

colors = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Value vs Transaction_Count
for k in range(K_OPTIMAL):
    mask = sample_km['prediction'] == k
    axes[0].scatter(
        sample_km[mask]['Transaction_Count'],
        sample_km[mask]['Value'],
        alpha=0.4, s=10, color=colors[k], label=f'Klaster {k}'
    )
axes[0].set_xlabel('Transaction Count')
axes[0].set_ylabel('Value (Rupee)')
axes[0].set_title(f'K-Means Clustering (k={K_OPTIMAL})\nValue vs Transaction Count', fontweight='bold')
axes[0].legend(markerscale=3)

# Plot 2: Distribusi klaster per domain
ct = sample_km.groupby(['Domain', 'prediction']).size().unstack(fill_value=0)
ct.plot(kind='bar', ax=axes[1], color=colors[:K_OPTIMAL], edgecolor='white')
axes[1].set_title('Distribusi Klaster per Domain', fontweight='bold')
axes[1].set_xlabel('Domain')
axes[1].set_ylabel('Jumlah Record (sample)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Klaster', fontsize=8)

plt.tight_layout()
plt.savefig('/content/ml_kmeans_clusters.png', dpi=150)
plt.show()

## 8. Perbandingan Semua Model

In [ ]:
# TABEL PERBANDINGAN SEMUA MODEL
results = {
    'Model': [
        'Linear Regression',
        'Random Forest Regressor',
        'Logistic Regression',
        'Random Forest Classifier',
        f'K-Means (k={K_OPTIMAL})'
    ],
    'Tipe': [
        'Regresi', 'Regresi',
        'Klasifikasi', 'Klasifikasi',
        'Clustering'
    ],
    'Metrik Utama': [
        f'R² = {r2_lr:.4f}',
        f'R² = {r2_rf:.4f}',
        f'Accuracy = {acc_lr*100:.2f}%',
        f'Accuracy = {acc_rfc*100:.2f}%',
        f'Silhouette = {silhouette:.4f}'
    ],
    'Metrik 2': [
        f'RMSE = {rmse_lr:,.0f}',
        f'RMSE = {rmse_rf:,.0f}',
        f'F1 = {f1_lr:.4f}',
        f'F1 = {f1_rfc:.4f}',
        f'k={K_OPTIMAL} klaster'
    ]
}

df_results = pd.DataFrame(results)
print('=' * 75)
print('RINGKASAN PERBANDINGAN SEMUA MODEL')
print('=' * 75)
print(df_results.to_string(index=False))
print('=' * 75)

In [ ]:
# Visualisasi Perbandingan Model
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Regresi: R² comparison
reg_models = ['Linear Regression', 'Random Forest\nRegressor']
reg_r2 = [r2_lr, r2_rf]
bars1 = axes[0].bar(reg_models, reg_r2, color=['#457B9D', '#2A9D8F'], edgecolor='white', width=0.5)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('R² Score')
axes[0].set_title('Perbandingan Model Regresi (R²)', fontweight='bold')
for bar, val in zip(bars1, reg_r2):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', fontweight='bold')

# Klasifikasi: Accuracy comparison
cls_models = ['Logistic\nRegression', 'Random Forest\nClassifier']
cls_acc = [acc_lr * 100, acc_rfc * 100]
bars2 = axes[1].bar(cls_models, cls_acc, color=['#E63946', '#E9C46A'], edgecolor='white', width=0.5)
axes[1].set_ylim(0, 110)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Perbandingan Model Klasifikasi (Accuracy)', fontweight='bold')
for bar, val in zip(bars2, cls_acc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.2f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/ml_model_comparison.png', dpi=150)
plt.show()

## 9. Kesimpulan & Wawasan ML

### Interpretasi Hasil:

**Model Regresi (Prediksi Value):**
- Random Forest Regressor diharapkan memberikan R² lebih tinggi dari Linear Regression karena mampu menangkap hubungan non-linear antar fitur.
- Fitur `Transaction_Count` dan `City_Avg_Value` umumnya menjadi prediktor terkuat.

**Model Klasifikasi (Prediksi Domain):**
- Random Forest Classifier umumnya lebih unggul dari Logistic Regression untuk data tabular multi-kelas.
- Accuracy tinggi mengindikasikan bahwa pola Value dan lokasi cukup unik per domain.

**K-Means Clustering:**
- Segmentasi transaksi membantu bank memahami pola perilaku nasabah tanpa label.
- Klaster nilai tinggi + transaksi tinggi = nasabah premium (target produk investasi).
- Klaster nilai rendah + transaksi tinggi = nasabah ritel aktif (target produk digital banking).

In [ ]:
# Simpan semua hasil ke CSV
df_results.to_csv('/content/ml_model_results.csv', index=False)
cluster_profile.toPandas().to_csv('/content/ml_cluster_profiles.csv', index=False)
print('Semua hasil tersimpan')

spark.stop()
print('SparkSession dihentikan')